# Exercise 1 — A Jupiter twin

*Covers notebooks 01–02. Difficulty: ★☆☆☆☆*

The simplest reflected-light target: a Jupiter twin around a Sun twin. How long does the ESC need, and what decides it?

In [1]:
import warnings
import numpy as np
import astropy.units as u
import matplotlib.pyplot as plt
import esc_etc
from esc_etc import Simulation, use_lazuli_style

use_lazuli_style()
warnings.filterwarnings("ignore", module="synphot")

def frame_for(sim, max_frame=1000 * u.s):
    """Frame time to use: max_frame, or half the saturation time if that is shorter."""
    return min(u.Quantity(max_frame, u.s), 0.5 * sim.saturation_time())


def hours_to_snr(sim, snr=5, max_frame=1000 * u.s):
    """Integration time (hours) to reach snr with the current configuration; NaN above the speckle plateau."""
    return sim.get_int_time_for_snr(snr, frame_for(sim, max_frame))["int_time_continuous"][0].to_value(u.hr)

### 1.1 — Jupiter at 10 pc

A Sun twin (V = 4.83) at 10 pc with a Jupiter twin at 5.2 AU: separation 0.52", contrast
$1.4 \times 10^{-9}$ (geometric albedo 0.5, Lambert phase at quadrature, $C = A_g (R_p/a)^2/\pi$).
With the R3 filter, raw contrast $10^{-8}$ and ppgain 10: what is the S/N after one hour of 1000 s
frames, and the time to S/N 5?

In [2]:
from astropy.constants import R_jup, au
C = 0.5 / np.pi * float((R_jup / (5.2 * au)) ** 2)
dm = -2.5 * np.log10(C)
sim = Simulation.from_default(filter="R3", hoststarmag=4.83, hoststar_band="johnson_v", planetdeltamag=dm, contrast=1e-8, ppgain=10, separation=0.52)
print(f"contrast {C:.2e} (delta mag {dm:.2f}): S/N in 1 h = {sim.get_snr(3600, 1000)['snr']:.2f}, time to S/N 5 = {hours_to_snr(sim):.1f} h")

contrast 1.34e-09 (delta mag 22.18): S/N in 1 h = 0.99, time to S/N 5 = nan h


> ✏️ **Before you run the next cell:** the plateau for this planet with ppgain 10 is 1.4. What ppgain is needed for S/N 5 to be reachable at all? Write your guess down (or put it in the room poll), then run.

### 1.2 — Post-processing gain

What is the plateau S/N for this planet at ppgain 10, 30 and 100, and the time to S/N 5 at each?

In [3]:
for g in (10, 30, 100):
    sim.update(coronagraph__ppgain=g)
    print(f"ppgain {g:3d}: plateau {g * C / 1e-8:5.1f}, time to S/N 5 = {hours_to_snr(sim):.1f} h")
sim.reset()

ppgain  10: plateau   1.3, time to S/N 5 = nan h


ppgain  30: plateau   4.0, time to S/N 5 = nan h


ppgain 100: plateau  13.4, time to S/N 5 = 13.2 h


### 1.3 — Which filter?

With ppgain 100, compute the time to S/N 5 in every filter (`esc_etc.FILTERS` has the names B1 to
B6 and R1 to R6). Which three are fastest, and why?

In [4]:
rows = []
for name in ["B1", "B2", "B3", "B4", "B5", "B6", "R1", "R2", "R3", "R4", "R5", "R6"]:
    s = Simulation.from_default(filter=name, hoststarmag=4.83, hoststar_band="johnson_v", planetdeltamag=dm, contrast=1e-8, ppgain=100, separation=0.52)
    rows.append((name, s.telescope.primary_filter.to_value(u.nm), hours_to_snr(s)))
for name, lam, h in sorted(rows, key=lambda r: r[2]):
    print(f"{name}: {lam:4.0f} nm, {h:7.1f} h")
# The wide filters (R3 = F659W, B4 = F450W) collect the most planet photons; the model has no colour, so only the bandwidth matters.

R3:  659 nm,    13.2 h
B4:  450 nm,   124.1 h
R2:  630 nm,   176.3 h
R1:  620 nm,   179.9 h
R6:  656 nm,   218.4 h
R4:  685 nm,   249.4 h
R5:  726 nm,   355.1 h
B6:  425 nm,   381.2 h
B3:  475 nm,  1136.7 h
B2:  450 nm,  2486.2 h
B5:  425 nm,  7525.9 h
B1:  400 nm, 555208.1 h


### 1.4 — Twice as far

Move the system to 20 pc: the contrast is unchanged, the separation halves, and the host is 1.5 mag
fainter. Time to S/N 5 in R3 at ppgain 100? Which of the three changes hurts most? (Test them one at
a time.)

In [5]:
sim = Simulation.from_default(filter="R3", hoststarmag=4.83, hoststar_band="johnson_v", planetdeltamag=dm, contrast=1e-8, ppgain=100, separation=0.52)
print(f"10 pc:                 {hours_to_snr(sim):.1f} h")
sim.update(coronagraph__separation=0.26); print(f"separation halved:     {hours_to_snr(sim):.1f} h"); sim.reset()
sim.update(scene__star__mag=4.83 + 1.5); print(f"host 1.5 mag fainter:  {hours_to_snr(sim):.1f} h"); sim.reset()
sim.update(coronagraph__separation=0.26, scene__star__mag=4.83 + 1.5); print(f"both (20 pc):          {hours_to_snr(sim):.1f} h"); sim.reset()

10 pc:                 13.2 h
separation halved:     19.2 h


host 1.5 mag fainter:  142.1 h
both (20 pc):          218.4 h


> 🤖 **Ask Jupyter AI:** *Add a loop over distance from 3 to 20 pc for the Jupiter twin, with the separation and host magnitude following, and plot the time to S/N 5 against distance for ppgain 30 and 100. Mark where the separation drops below the IWA from notebook 03.*
>
> Then ask it where Saturn (9.5 AU, albedo 0.5) would sit on the same plot.

***
## Share-back and feedback

Bring back **one plot and one number** to the share-back board, and the observing decision you draw
from them.

> 🤖 **Ask Jupyter AI:** *Draft a GitHub issue for schmidt-observatories/esc-etc describing one thing I
> found missing, confusing or wrong in this exercise, with a minimal code example.*